In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [23]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": { "temperature": temperature },
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [24]:
# calc tool function
import ast
import operator
from anthropic.types import ToolParam

_BINARY_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}
_UNARY_OPS = {ast.UAdd: operator.pos, ast.USub: operator.neg}


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _BINARY_OPS:
        return _BINARY_OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY_OPS:
        return _UNARY_OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("Unsupported expression")


def calculator(expression):
    # Parse instead of eval() so only plain arithmetic can run
    tree = ast.parse(expression.strip(), mode="eval")
    return _eval_node(tree.body)


calculator_schema = ToolParam(
    {
        "name": "calculator",
        "description": (
            "Evaluates a basic arithmetic expression and returns the numeric result. "
            "Use it for any math instead of calculating in your head, especially with "
            "large numbers, decimals, or multiple operations. "
            "Supports +, -, *, /, // (floor division), % (modulo), ** (exponent), "
            "unary minus, and parentheses, following standard operator precedence. "
            "Does not support variables, functions (e.g. sqrt, sin), or units; "
            "it cannot do date math, so convert dates to plain numbers first. "
            "Returns an error for invalid syntax or division by zero."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": (
                        "The arithmetic expression to evaluate, written as plain text "
                        "using only numbers, operators and parentheses. "
                        "Examples: '177 * 24', '(1500 + 250) / 4', '2 ** 10'."
                    ),
                }
            },
            "required": ["expression"],
        },
    }
)

In [25]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
}

In [26]:
import json


def run_tool(tool_name, tool_input):
    if tool_name == "calculator":
        return calculator(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [33]:
import logging

logger = logging.getLogger(__name__)

# Safety bound only; normal runs end via stop_reason well before this
MAX_ITERATIONS = 20


def run_conversation(messages):
    iteration = 0

    while iteration < MAX_ITERATIONS:
        iteration += 1

        response = chat(
            messages,
            tools=[
                calculator_schema,
                web_search_schema,
            ],
        )

        add_assistant_message(messages, response)
        print(text_from_message(response))
        print(f"iteration: {iteration}")

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    else:
        # while-else: runs only if the loop was never broken out of
        logger.warning(
            "run_conversation hit MAX_ITERATIONS (%d) without a final response; stopping.",
            MAX_ITERATIONS,
        )

    return messages

In [34]:
messages = []
add_user_message(
    messages,
    "check these 2 things: the distance from earth to the sun and the distance from earth to the moon, rest the 2 values to see whats the difference",
)
run_conversation(messages)

run_conversation hit MAX_ITERATIONS (1) without a final response; stopping.



iteration: 1


[{'role': 'user',
  'content': 'check these 2 things: the distance from earth to the sun and the distance from earth to the moon, rest the 2 values to see whats the difference'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01FjhDGLF7V776uZPHRd4t1Y', caller=DirectCaller(type='direct'), input={'expression': '149600000 - 384400'}, name='calculator', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01FjhDGLF7V776uZPHRd4t1Y',
    'content': '149215600',
    'is_error': False}]}]